In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
from google.colab import drive
drive.mount ('/content/drive')

In [ ]:
dataset_path='/content/drive/MyDrive/Student Depression Dataset.csv'

In [ ]:
df = pd.read_csv(dataset_path)

In [ ]:
df.head()

In [ ]:
df


In [ ]:
df.shape

**Phase 1: securing input fidelity**

In [ ]:
# Filter only numeric columns
numeric_columns = df.select_dtypes(include=['number'])

# Calculate the standard deviation for all numeric columns
std_deviation = numeric_columns.std()

# Print the result
print("Standard deviation of all numeric columns:")
print(std_deviation)

In [ ]:
def missing_value_report(df: pd.DataFrame) -> pd.Series:
    """Return missingness proportion per column, sorted descending."""
    return (df.isna().sum() / len(df)).sort_values(ascending=False)

In [ ]:
def handle_missing_values(df: pd.DataFrame, group_col: str = None) -> pd.DataFrame:
    """
    Applies the Missing Data Decision Matrix from the training deck:

        < 5%      -> Drop rows   (dropna) — preserves distribution, avoids bias
        5% - 20%  -> Statistical imputation
                       - numeric & skewed   -> median
                       - categorical        -> mode (or group-wise mode if
                                                group_col is provided)
        > 20%     -> KNN Imputation (captures multi-dimensional relationships)
    """
    df = df.copy()
    missing_pct = missing_value_report(df)

    for col in df.columns:
        pct = missing_pct.get(col, 0)
        if pct == 0:
            continue

        if pct < 0.05:
            # Low missingness: safe to drop just those rows
            df = df.dropna(subset=[col])

        elif pct <= 0.20:
            if pd.api.types.is_numeric_dtype(df[col]):
                if group_col and group_col in df.columns:
                    df[col] = df.groupby(group_col)[col].transform(
                        lambda s: s.fillna(s.median())
                    )
                    df[col] = df[col].fillna(df[col].median())  # fallback
                else:
                    df[col] = df[col].fillna(df[col].median())
            else:
                if group_col and group_col in df.columns:
                    mode_fill = df.groupby(group_col)[col].transform(
                        lambda s: s.fillna(s.mode().iloc[0] if not s.mode().empty else s)
                    )
                    df[col] = mode_fill
                else:
                    mode_val = df[col].mode()
                    df[col] = df[col].fillna(mode_val.iloc[0] if not mode_val.empty else "Unknown")

        else:  # > 20% missing -> KNN Imputation
            from sklearn.impute import KNNImputer
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            if col in numeric_cols:
                imputer = KNNImputer(n_neighbors=5)
                df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
            else:
                # KNN doesn't work directly on categoricals; fall back to mode
                mode_val = df[col].mode()
                df[col] = df[col].fillna(mode_val.iloc[0] if not mode_val.empty else "Unknown")

    return df

In [ ]:
def get_iqr_bounds(series: pd.Series) -> tuple[float, float]:
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

In [ ]:
def neutralize_outliers(df: pd.DataFrame, columns=None, method: str = "iqr") -> pd.DataFrame:
    """
    Neutralizes outliers via WINSORIZATION (clipping), not deletion —
    this preserves row count and sequential integrity (per the deck's
    'Winsorization vs. Deletion' guidance).

    method: "iqr" (default) or "zscore"
    """
    df = df.copy()
    columns = columns or df.select_dtypes(include=[np.number]).columns.tolist()

    for col in columns:
        if method == "iqr":
            lower, upper = get_iqr_bounds(df[col])
        elif method == "zscore":
            mean, std = df[col].mean(), df[col].std()
            lower, upper = mean - 3 * std, mean + 3 * std
        else:
            raise ValueError("method must be 'iqr' or 'zscore'")

        df[col] = np.clip(df[col], lower, upper)  # vectorized, no loops

    return df

# PHASE 2 — THE VECTORIZED COMPUTATION ENGINE

In [ ]:
def one_hot_encode(df: pd.DataFrame, categorical_cols=None) -> pd.DataFrame:
    """
    One-Hot Encoding (the 'fix') instead of Label Encoding (the 'flaw').
    Label encoding nominal categories creates a false ordinal distance
    (e.g. Tokyo=3x London); one-hot maps C classes to C orthogonal axes.
    """
    categorical_cols = categorical_cols or df.select_dtypes(include=["object", "category"]).columns.tolist()
    return pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
plt.figure(figsize=(7,5))
sns.countplot(data=df, x='Gender', hue='Depression')

plt.title('Depression Status by Gender')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.legend(title='Depression')
plt.show()

In [ ]:
import seaborn as sns
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='Sleep Duration', hue='Depression')

plt.title('Depression by Sleep Duration')
plt.xlabel('Sleep Duration')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

table = pd.crosstab(df['Gender'], df['Depression'], normalize='index')*100

table.plot(kind='bar')

plt.title('Percentage of Depression by Gender')
plt.xlabel('Gender')
plt.ylabel('Percentage (%)')
plt.legend(title='Depression')
plt.show()

In [ ]:
depression_counts = df['Depression'].value_counts()

plt.figure(figsize=(6,6))
plt.pie(depression_counts,
        labels=depression_counts.index,
        autopct='%1.1f%%')

plt.title('Depression Prevalence')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(df['CGPA'], bins=20, kde=True)

plt.title('Distribution of CGPA')
plt.xlabel('CGPA')
plt.ylabel('Frequency')

plt.show()

In [ ]:
plt.figure(figsize=(7,5))

sns.boxplot(
    data=df,
    x='Depression',
    y='Age'
)

plt.title('Age by Depression Status')
plt.xlabel('Depression')
plt.ylabel('Age')

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df,
    x='Academic Pressure',
    y='CGPA'
)

plt.title('Academic Pressure vs CGPA')

plt.show()

In [ ]:
import numpy as np

numeric_vars = [
    'Age',
    'Academic Pressure',
    'Work Pressure',
    'CGPA',
    'Study Satisfaction',
    'Job Satisfaction',
    'Work/Study Hours',
    'Financial Stress'
]

corr = df[numeric_vars].corr()

plt.figure(figsize=(8,6))

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Matrix')

plt.show()

## Handle Missing Values

In [ ]:
print('Missing values before handling:')
display(missing_value_report(df))

In [ ]:
df_cleaned = handle_missing_values(df)
print('\nMissing values after handling:')
display(missing_value_report(df_cleaned))

In [ ]:
print('\nDataFrame Info after handling missing values:')
df_cleaned.info()

## Categorical Feature Encoding

In [ ]:
print('Unique values for categorical columns in df_cleaned:')
for col in df_cleaned.select_dtypes(include=['object']).columns:
    print(f'{col}: {df_cleaned[col].nunique()} unique values')
    if df_cleaned[col].nunique() <= 10:  # Display unique values for columns with few unique values
        print(f'  Unique values: {df_cleaned[col].unique()}')
    else:
        print(f'  (Too many unique values to display all)')

### Label Encoding for Binary Features

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Identify binary columns
binary_cols = ['Gender', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']

# Apply Label Encoding
for col in binary_cols:
    le = LabelEncoder()
    df_cleaned[col] = le.fit_transform(df_cleaned[col])
    print(f'Column \'{col}\' encoded. Mappings: {list(le.classes_)} -> {list(range(len(le.classes_)))}')

df_cleaned.head()

### One-Hot Encoding for Low Cardinality Nominal Features

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Identify low cardinality nominal columns
low_cardinality_cols = ['Sleep Duration', 'Dietary Habits']

# Apply One-Hot Encoding
df_encoded = pd.get_dummies(df_cleaned, columns=low_cardinality_cols, prefix=low_cardinality_cols)

print("Original columns: ", df_cleaned.columns.tolist())
print("New columns after One-Hot Encoding: ", df_encoded.columns.tolist())
df_encoded.head()

### Target Encoding for High Cardinality Nominal Features

In [ ]:
!pip install category_encoders

In [ ]:
from category_encoders import TargetEncoder

# Identify high cardinality nominal columns
high_cardinality_cols = ['City', 'Profession', 'Degree']

# Apply Target Encoding
for col in high_cardinality_cols:
    encoder = TargetEncoder(cols=[col])
    df_encoded[col] = encoder.fit_transform(df_encoded[col], df_encoded['Depression'])
    print(f'Column \'{col}\' target encoded.')

print('\nDataFrame after Target Encoding:')
df_encoded.head()

## Final Data Preparation

In [ ]:
print('DataFrame Info after all encoding steps:')
df_encoded.info()

In [ ]:
# Drop the 'id' column as it's an identifier and not a feature
df_processed = df_encoded.drop(columns=['id'])

print('\nDataFrame after dropping \'id\' column:')
display(df_processed.head())

In [ ]:
# Apply outlier neutralization to the processed DataFrame
df_processed_no_outliers = neutralize_outliers(df_processed, method='iqr')

print('DataFrame after outlier neutralization:')
display(df_processed_no_outliers.head())

In [ ]:

from sklearn.preprocessing import StandardScaler

# Exclude 'Depression' column from scaling as it is the target variable
X_no_outliers = df_processed_no_outliers.drop('Depression', axis=1)
y_no_outliers = df_processed_no_outliers['Depression']

# Identify numerical columns for scaling
# Boolean columns are implicitly handled by some scalers or can be excluded if desired
numerical_cols_to_scale = X_no_outliers.select_dtypes(include=['float64', 'int64']).columns

# Initialize the StandardScaler
scaler = StandardScaler()

# Apply scaling to the numerical columns
X_scaled = X_no_outliers.copy()
X_scaled[numerical_cols_to_scale] = scaler.fit_transform(X_no_outliers[numerical_cols_to_scale])

print('DataFrame after Feature Scaling (first 5 rows):')
display(X_scaled.head())

## Splitting Data into Training and Testing Sets

In [ ]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df_processed.drop('Depression', axis=1)
y = df_processed['Depression']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print('\nClass distribution in y_train:')
print(y_train.value_counts(normalize=True))

print('\nClass distribution in y_test:')
print(y_test.value_counts(normalize=True))

## Model Training and Evaluation: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Initialize and train the Logistic Regression model
log_reg_model = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' solver works well for small datasets and binary classification
log_reg_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = log_reg_model.predict(X_test)
y_pred_proba = log_reg_model.predict_proba(X_test)[:, 1]

# Evaluate the model
print("Logistic Regression Model Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

# Display Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Visualize Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])
plt.title('Confusion Matrix for Logistic Regression')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()